# 05 — Run ML

Train the reproducible three-cluster urban typology, log its model and metrics with MLflow, and persist cell predictions, cluster profiles, and district cluster composition as governed Gold tables.

## 1. KMeans contract

The model follows the reviewed legacy contract: simplified land use is one-hot encoded, mean and maximum building height plus public-transport stop count are standardized, and categorical and numeric groups are balanced before KMeans. Only canonically mapped cells with all four inputs are eligible. All 10,100 cells remain in the prediction table with an explicit eligibility status.

The fixed configuration uses three clusters, random state 42, and 20 initializations. Cluster labels describe exploratory urban patterns rather than objective neighborhood rankings.

In [ ]:
import importlib
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import mlflow.sklearn

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

for module_name in [
    "urban_workflow.modeling.table_operations",
    "urban_workflow.modeling.clustering",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

from urban_workflow.modeling.clustering import (
    CATEGORICAL_FEATURES, CLUSTER_COUNT, N_INIT, NUMERIC_FEATURES,
    RANDOM_STATE, fit_kmeans_typology,
)
from urban_workflow.modeling.anomaly_detection import (
    ANOMALY_FEATURE_COLUMNS, CONTAMINATION, N_ESTIMATORS,
    RANDOM_STATE as ANOMALY_RANDOM_STATE, fit_isolation_forest,
)
from urban_workflow.modeling.table_operations import (
    build_anomaly_output_frame, build_typology_output_frames,
    validate_anomaly_output, validate_typology_outputs,
    write_snapshot_table,
)

print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
if not all((CATALOG, SCHEMA)):
    raise ValueError("catalog and schema must be non-empty.")
GOLD_GRID_TABLE = f"{CATALOG}.{SCHEMA}.gold_grid_features"
GRID_TYPOLOGY_TABLE = f"{CATALOG}.{SCHEMA}.gold_grid_typology"
CLUSTER_PROFILE_TABLE = f"{CATALOG}.{SCHEMA}.gold_cluster_profiles"
DISTRICT_CLUSTER_MIX_TABLE = f"{CATALOG}.{SCHEMA}.gold_district_cluster_mix"
GOLD_DISTRICT_TABLE = f"{CATALOG}.{SCHEMA}.gold_district_features"
DISTRICT_ANOMALY_TABLE = f"{CATALOG}.{SCHEMA}.gold_district_anomalies"

model_started_at = datetime.now(timezone.utc)
model_run_id = (
    f"kmeans_{model_started_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Model run: {model_run_id}")

## 2. Prepare, train, and log

The grid is driver-sized, so the four modeling inputs and identifiers are collected to Pandas for the scikit-learn pipeline. MLflow stores the fixed parameters, silhouette score, fitted preprocessor, fitted KMeans model, and final encoded feature names.

In [ ]:
source_grid = spark.table(GOLD_GRID_TABLE)
model_input_columns = [
    "cell_id", "district_code", "district_name",
    "district_mapping_status",
    *CATEGORICAL_FEATURES, *NUMERIC_FEATURES,
]
model_input = source_grid.select(*model_input_columns).toPandas()

with mlflow.start_run(run_name=model_run_id) as active_run:
    artifacts = fit_kmeans_typology(model_input)
    mlflow_run_id = active_run.info.run_id
    mlflow.log_params({
        "cluster_count": CLUSTER_COUNT,
        "random_state": RANDOM_STATE,
        "n_init": N_INIT,
        "input_rows": len(model_input),
        "eligible_rows": artifacts.eligible_count,
        "feature_group_balancing": True,
    })
    mlflow.log_metric("silhouette_score", artifacts.silhouette)
    mlflow.log_dict(
        {
            "categorical_features": list(CATEGORICAL_FEATURES),
            "numeric_features": list(NUMERIC_FEATURES),
            "encoded_feature_names": artifacts.feature_names,
        },
        "feature_contract.json",
    )
    mlflow.sklearn.log_model(artifacts.preprocessor, artifact_path="preprocessor")
    mlflow.sklearn.log_model(artifacts.model, artifact_path="kmeans_model")

print({
    "mlflow_run_id": mlflow_run_id,
    "eligible_rows": artifacts.eligible_count,
    "silhouette_score": artifacts.silhouette,
    "encoded_feature_count": len(artifacts.feature_names),
})

## 3. Build and validate outputs

The cell table retains every source row. Profiles summarize the three clusters, and district composition records the share of eligible cells assigned to each cluster. Every district share vector must sum to one.

In [ ]:
grid_typology, cluster_profiles, district_cluster_mix = build_typology_output_frames(
    spark,
    source_grid,
    artifacts,
    model_run_id=model_run_id,
    mlflow_run_id=mlflow_run_id,
    trained_at_utc=model_started_at,
)
typology_validation = validate_typology_outputs(
    source_grid, grid_typology, cluster_profiles, district_cluster_mix
)
print(typology_validation)
display(cluster_profiles.orderBy("cluster_label"))
display(district_cluster_mix.orderBy("district_code"))

## 4. Persist and read back

The three fixed outputs are overwritten together and read back through the same acceptance gate. Their model and MLflow run IDs make every row traceable to the fitted artifacts.

In [ ]:
write_snapshot_table(grid_typology, GRID_TYPOLOGY_TABLE)
write_snapshot_table(cluster_profiles, CLUSTER_PROFILE_TABLE)
write_snapshot_table(district_cluster_mix, DISTRICT_CLUSTER_MIX_TABLE)
spark.sql(f"COMMENT ON TABLE {GRID_TYPOLOGY_TABLE} IS 'KMeans eligibility and urban-typology assignments for all Gold grid cells.'")
spark.sql(f"COMMENT ON TABLE {CLUSTER_PROFILE_TABLE} IS 'Interpretable profiles for the three KMeans urban typology clusters.'")
spark.sql(f"COMMENT ON TABLE {DISTRICT_CLUSTER_MIX_TABLE} IS 'District shares of eligible grid cells by KMeans urban typology cluster.'")

persisted_grid_typology = spark.table(GRID_TYPOLOGY_TABLE)
persisted_cluster_profiles = spark.table(CLUSTER_PROFILE_TABLE)
persisted_district_cluster_mix = spark.table(DISTRICT_CLUSTER_MIX_TABLE)
persisted_validation = validate_typology_outputs(
    source_grid,
    persisted_grid_typology,
    persisted_cluster_profiles,
    persisted_district_cluster_mix,
)
print(persisted_validation)
print(
    f"Validated {persisted_grid_typology.count()} grid typology rows, "
    f"{persisted_cluster_profiles.count()} cluster profiles, and "
    f"{persisted_district_cluster_mix.count()} district cluster-mix rows."
)

## 5. District mismatch detection

IsolationForest combines the accepted district features with the current KMeans cluster shares. Missing numeric inputs are median-imputed and all inputs are standardized. With only 21 districts, its flags are exploratory relative-mismatch signals. The three explanation features are the largest absolute standardized deviations; they are descriptive rather than causal.

In [ ]:
district_source = spark.table(GOLD_DISTRICT_TABLE)
district_model_input = district_source.select(
    "district_code", "district_name",
    *[name for name in ANOMALY_FEATURE_COLUMNS if not name.startswith("cluster_share_")],
).toPandas()
cluster_mix_input = persisted_district_cluster_mix.select(
    "district_code",
    *[name for name in ANOMALY_FEATURE_COLUMNS if name.startswith("cluster_share_")],
).toPandas()
anomaly_started_at = datetime.now(timezone.utc)
anomaly_model_run_id = (
    f"isolation_forest_{anomaly_started_at.strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

with mlflow.start_run(run_name=anomaly_model_run_id) as anomaly_run:
    anomaly_artifacts = fit_isolation_forest(
        district_model_input, cluster_mix_input
    )
    anomaly_mlflow_run_id = anomaly_run.info.run_id
    mlflow.log_params({
        "contamination": CONTAMINATION,
        "random_state": ANOMALY_RANDOM_STATE,
        "n_estimators": N_ESTIMATORS,
        "input_districts": len(district_model_input),
        "eligible_districts": anomaly_artifacts.eligible_count,
        "imputation": "median",
        "scaling": "standard",
    })
    mlflow.log_metric("flagged_district_count", anomaly_artifacts.flagged_count)
    mlflow.log_dict(
        {"feature_columns": anomaly_artifacts.feature_columns},
        "anomaly_feature_contract.json",
    )
    mlflow.sklearn.log_model(
        anomaly_artifacts.preprocessor, artifact_path="preprocessor"
    )
    mlflow.sklearn.log_model(
        anomaly_artifacts.model, artifact_path="isolation_forest_model"
    )

district_anomalies = build_anomaly_output_frame(
    spark,
    district_source,
    persisted_district_cluster_mix,
    anomaly_artifacts,
    model_run_id=anomaly_model_run_id,
    mlflow_run_id=anomaly_mlflow_run_id,
    trained_at_utc=anomaly_started_at,
)
anomaly_validation = validate_anomaly_output(
    district_source, district_anomalies
)
print(anomaly_validation)
display(
    district_anomalies.select(
        "anomaly_rank", "district_code", "district_name",
        "anomaly_score", "anomaly_flag",
        "anomaly_top_features", "anomaly_interpretation_notes",
    ).orderBy("anomaly_rank")
)

## 6. Persist anomaly results

The 21-row output retains the complete district feature context, KMeans shares, ranked IsolationForest results, descriptive explanations, and the district, KMeans, IsolationForest, and MLflow run lineage.

In [ ]:
write_snapshot_table(district_anomalies, DISTRICT_ANOMALY_TABLE)
spark.sql(
    f"COMMENT ON TABLE {DISTRICT_ANOMALY_TABLE} IS "
    "'Exploratory IsolationForest district mismatch scores with descriptive feature explanations.'"
)
persisted_district_anomalies = spark.table(DISTRICT_ANOMALY_TABLE)
persisted_anomaly_validation = validate_anomaly_output(
    district_source, persisted_district_anomalies
)
print(persisted_anomaly_validation)
print(
    f"Validated {persisted_district_anomalies.count()} district anomaly rows "
    f"with {anomaly_artifacts.flagged_count} exploratory flags."
)
dbutils.notebook.exit(
    f"SUCCESS: Validated 10,100 typology rows, 3 cluster profiles, "
    f"21 district mixes, and 21 anomaly rows with "
    f"{anomaly_artifacts.flagged_count} flags."
)

## 7. Slice status

The modeling milestone is complete when KMeans and IsolationForest both have MLflow artifacts and all four Gold outputs pass live read-back validation. Notebook 06 performs the explicit review and legacy-comparison step.